In [37]:
# =============================================================================
# CELL 0: Imports
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings

In [38]:

# =============================================================================
# CELL 1: Configuration and Setup (Production Mix Data)
# =============================================================================

NOTEBOOK_NAME = "04: Production Mix Data Ingestion"
DATA_SUBFOLDER = "production_mix"
OBJECTIVE = """Process E-Control Austria electricity production mix data
Monthly electricity production and consumption by source
18 variables including hydropower, fossil fuels, renewables, imports/exports

IMPORTANT ARCHITECTURAL DECISION:
Production mix data is MONTHLY ONLY - no daily or weekly rows will be created.
Consistent with climate data architecture to prevent empty rows in consolidated dataset.
Production variables can only be used in monthly-level analyses.

CRITICAL: Column selection uses STRING column names ('15', '16', etc.) not numeric indices.
The source file has complex 14-row header structure."""

# Production Mix specific configuration
PRODUCTION_FILE_NAME = "el_dataset_mn.csv"
DATE_COLUMN = "Header & Timestamp"
DELIMITER = ";"  # Semicolon delimiter in European CSVs
DECIMAL_SEPARATOR = ","
SKIPROWS = range(1, 14)  # Skip rows 2-14 (complex header metadata)

# Column mapping: Source column name → Target column name
# Note: Column names are STRINGS ('15', '16', etc.) not numeric indices
COLUMN_MAPPING = {
    'Header & Timestamp': 'date',
    '15': 'prod_gross_electricity_production',
    '16': 'prod_gross_electricity_consumption',
    '37': 'prod_hydropower_production_total',
    '38': 'prod_fossil_sk_production',
    '39': 'prod_fossil_DvfB_production',
    '40': 'prod_fossil_DvOe_production',
    '41': 'prod_fossil_gas_production',
    '42': 'prod_fossil_subtotal_production',
    '43': 'prod_renewable_bio_production',
    '44': 'prod_renewable_SoBio_production',
    '45': 'prod_other_fuels_production',
    '46': 'prod_fuel_production_total',
    '47': 'prod_wind_total',
    '48': 'prod_pv_total',
    '49': 'prod_geothermal_total',
    '51': 'prod_power_production_total',
    '57': 'prod_electricity_imports',
    '58': 'prod_electricity_exports'
}

# Setup functions (reused from electricity/carbon/climate - consistent)
def setup_pandas_options():
    """Configure pandas display options for better data inspection."""
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', 100)  
    pd.set_option('display.width', None)
    warnings.filterwarnings('ignore')

def setup_project_paths(data_subfolder):
    """Set up project directory paths for any data source."""
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
    RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / data_subfolder
    PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed'
    PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
    return PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH

def print_notebook_header(notebook_name, objective, raw_path, processed_path):
    """Print standardized notebook start information."""
    print("="*60)
    print(f"NOTEBOOK: {notebook_name}")
    print("="*60)
    print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Raw Data Path: {raw_path}")
    print(f"Processed Data Path: {processed_path}")
    print(f"Objective: {objective}")
    print("="*60)

# Setup execution
setup_pandas_options()
PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH = setup_project_paths(DATA_SUBFOLDER)
print_notebook_header(NOTEBOOK_NAME, OBJECTIVE, RAW_DATA_PATH, PROCESSED_DATA_PATH)

print("Cell 1: Configuration and Setup - READY")

NOTEBOOK: 04: Production Mix Data Ingestion
Start Time: 2025-10-01 06:53:16
Raw Data Path: c:\Users\paulr\Desktop\Capstone\data\raw\production_mix
Processed Data Path: c:\Users\paulr\Desktop\Capstone\data\processed
Objective: Process E-Control Austria electricity production mix data
Monthly electricity production and consumption by source
18 variables including hydropower, fossil fuels, renewables, imports/exports

IMPORTANT ARCHITECTURAL DECISION:
Production mix data is MONTHLY ONLY - no daily or weekly rows will be created.
Consistent with climate data architecture to prevent empty rows in consolidated dataset.
Production variables can only be used in monthly-level analyses.

CRITICAL: Column selection uses STRING column names ('15', '16', etc.) not numeric indices.
The source file has complex 14-row header structure.
Cell 1: Configuration and Setup - READY


In [39]:
# =============================================================================
# CELL 2: Missing Values Standardization Function (with Quality Control)
# =============================================================================

def standardize_missing_values(df, additional_missing=None, show_quality_control=True):
    """
    Convert various missing value representations to pandas NaN.
    
    Args:
        df (DataFrame): Input dataframe
        additional_missing (list): Extra missing indicators beyond defaults
        show_quality_control (bool): Whether to display quality control output
    
    Returns:
        DataFrame: Dataframe with standardized missing values
        dict: Report of found missing patterns
    """
    missing_indicators = [
        'N/A', 'n/a', 'NA', 'na',
        '', '-', '--', '---',
        'NULL', 'null', 'Null',
        'NaN', 'nan', '#N/A'
    ]
    
    if additional_missing:
        missing_indicators.extend(additional_missing)
    
    if show_quality_control:
        print("\nMISSING VALUES STANDARDIZATION - QUALITY CONTROL:")
        print("-" * 55)
        print("Missing indicators standardized:")
        print("  ", ', '.join([f"'{ind}'" for ind in missing_indicators]))
        print()
    
    found_patterns = {}
    df_clean = df.copy()
    
    for col in df_clean.columns:
        original_nulls = df_clean[col].isnull().sum()
        df_clean[col] = df_clean[col].replace(missing_indicators, np.nan)
        new_nulls = df_clean[col].isnull().sum()
        converted_count = new_nulls - original_nulls
        
        if converted_count > 0:
            found_patterns[col] = {
                'original_nulls': original_nulls,
                'converted_missing': converted_count,
                'total_nulls': new_nulls
            }
    
    if show_quality_control:
        if found_patterns:
            print("CONVERSION RESULTS BY COLUMN:")
            for col, pattern in found_patterns.items():
                print(f"  {col}:")
                print(f"    Original nulls: {pattern['original_nulls']}")
                print(f"    Converted missing: {pattern['converted_missing']}")
                print(f"    Total nulls: {pattern['total_nulls']}")
        else:
            print("No missing value patterns found for conversion")
        print("-" * 55)
    
    return df_clean, found_patterns

print("Cell 2: Missing Values Standardization Function - READY")

Cell 2: Missing Values Standardization Function - READY


In [40]:
# =============================================================================
# CELL 3: Production Mix File Discovery
# =============================================================================

def discover_production_file(data_path, filename):
    """
    Discover single production mix data file.
    
    Args:
        data_path (Path): Path to raw data directory
        filename (str): Expected filename
    
    Returns:
        dict: File information dictionary
    """
    file_path = data_path / filename
    
    if file_path.exists():
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': True
        }
    else:
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': False
        }

def display_file_discovery_results(file_info):
    """Display results of production mix file discovery."""
    print("Production mix file discovery:")
    print(f"  File: {file_info['filename']}")
    print(f"  Exists: {file_info['exists']}")
    print(f"  Path: {file_info['path']}")
    
    if file_info['exists']:
        print("Ready to proceed with production mix data loading")
    else:
        print("ERROR: Production mix file not found!")

# Execute file discovery
production_file_info = discover_production_file(RAW_DATA_PATH, PRODUCTION_FILE_NAME)
display_file_discovery_results(production_file_info)

print("Cell 3: Production Mix File Discovery - COMPLETE")

Production mix file discovery:
  File: el_dataset_mn.csv
  Exists: True
  Path: c:\Users\paulr\Desktop\Capstone\data\raw\production_mix\el_dataset_mn.csv
Ready to proceed with production mix data loading
Cell 3: Production Mix File Discovery - COMPLETE


In [41]:
# =============================================================================
# CELL 4: Load and Clean Production Mix Data
# =============================================================================

def load_production_data(file_path, skiprows, delimiter, column_mapping):
    """
    Load production mix data and select relevant columns only.
    
    Args:
        file_path (str/Path): Path to production CSV file
        skiprows (range): Row indices to skip (complex header metadata)
        delimiter (str): File delimiter
        column_mapping (dict): Mapping of source to target column names
    
    Returns:
        tuple: (dataframe, metadata)
    """
    try:
        # Load dataset with header skip
        df = pd.read_csv(
            file_path, 
            delimiter=delimiter, 
            skiprows=skiprows,
            decimal=DECIMAL_SEPARATOR,
            encoding='ISO-8859-1'  # Handle special characters in European data
        )
        
        print(f"INITIAL DATA LOAD:")
        print(f"  Original shape: {df.shape}")
        print(f"  Rows after header skip: {len(df)}")
        print(f"  Total columns available: {len(df.columns)}")
        print()
        
        # Select only relevant columns (18 production variables + date)
        source_columns = list(column_mapping.keys())
        
        print(f"COLUMN SELECTION:")
        print(f"  Selecting {len(source_columns)} of {len(df.columns)} columns")
        print(f"  Source columns (STRING names): {source_columns[:5]}... (showing first 5)")
        print()
        
        # Column selection using STRING column names
        df_filtered = df[source_columns].copy()
        
        print(f"After column selection: {df_filtered.shape}")
        print()
        
        # Clean missing values BEFORE renaming (to show original column names in report)
        df_clean, missing_patterns = standardize_missing_values(df_filtered, show_quality_control=True)
        
        # Rename columns to standardized names
        df_clean.rename(columns=column_mapping, inplace=True)
        
        print(f"\nCOLUMN RENAMING:")
        print(f"  Renamed {len(column_mapping)} columns with prod_ prefix")
        print(f"  New columns: {df_clean.columns.tolist()}")
        print()
        
        # Convert date column to datetime
        # Try to parse flexibly (format may vary)
        df_clean['date'] = pd.to_datetime(df_clean['date'], format='%Y-%m')
        
        # Create metadata (consistent with other pipelines)
        metadata = {
            'filename': Path(file_path).name,
            'rows': len(df_clean),
            'columns': list(df_clean.columns),
            'missing_patterns_found': missing_patterns,
            'date_range': (df_clean['date'].min(), df_clean['date'].max()) if len(df_clean) > 0 else (None, None)
        }
        
        print(f"DATA SUMMARY:")
        print(f"  Final shape: {df_clean.shape}")
        print(f"  Date range: {metadata['date_range'][0].strftime('%Y-%m-%d')} to {metadata['date_range'][1].strftime('%Y-%m-%d')}")
        print()
        
        print("SAMPLE DATA:")
        print(df_clean.head(10))
        
        return df_clean, metadata
        
    except Exception as e:
        print(f"ERROR loading production data: {e}")
        return None, {'error': str(e)}

# Execute data loading
if production_file_info['exists']:
    production_df, production_metadata = load_production_data(
        production_file_info['path'],
        SKIPROWS,
        DELIMITER,
        COLUMN_MAPPING
    )
    
    if production_df is None:
        print(f"ERROR loading production data: {production_metadata['error']}")
else:
    print("Skipping data load - file not found")

print("Cell 4: Load and Clean Production Mix Data - COMPLETE")

INITIAL DATA LOAD:
  Original shape: (187, 67)
  Rows after header skip: 187
  Total columns available: 67

COLUMN SELECTION:
  Selecting 19 of 67 columns
  Source columns (STRING names): ['Header & Timestamp', '15', '16', '37', '38']... (showing first 5)

After column selection: (187, 19)


MISSING VALUES STANDARDIZATION - QUALITY CONTROL:
-------------------------------------------------------
Missing indicators standardized:
   'N/A', 'n/a', 'NA', 'na', '', '-', '--', '---', 'NULL', 'null', 'Null', 'NaN', 'nan', '#N/A'

No missing value patterns found for conversion
-------------------------------------------------------

COLUMN RENAMING:
  Renamed 19 columns with prod_ prefix
  New columns: ['date', 'prod_gross_electricity_production', 'prod_gross_electricity_consumption', 'prod_hydropower_production_total', 'prod_fossil_sk_production', 'prod_fossil_DvfB_production', 'prod_fossil_DvOe_production', 'prod_fossil_gas_production', 'prod_fossil_subtotal_production', 'prod_renewable_bio_

In [42]:
# =============================================================================
# CELL 5: Convert Data Types and Create Long Format
# =============================================================================

def convert_production_data_types(df):
    """
    Convert production columns to appropriate numeric types.
    Production values are rounded to integers (Int64), which will become float64 in CSV.
    
    Args:
        df (DataFrame): Production data with prod_ prefixed columns
    
    Returns:
        DataFrame: Data with converted types
    """
    df_converted = df.copy()
    
    print("DATA TYPE CONVERSION:")
    print("-" * 40)
    
    # Get all production columns (exclude date column)
    prod_columns = [col for col in df_converted.columns if col.startswith('prod_')]
    
    print(f"Converting {len(prod_columns)} production columns to numeric:")
    
    for col in prod_columns:
        # Convert to numeric first (handles any string values)
        df_converted[col] = pd.to_numeric(df_converted[col], errors='coerce')
        
        # Round to integers
        df_converted[col] = df_converted[col].round(0)
        
        # Convert to Int64 (nullable integer, will become float64 in CSV)
        df_converted[col] = df_converted[col].astype('Int64')
        
        print(f"  {col}: Int64")
    
    print()
    print("NOTE: Int64 types will convert to float64 when saved to CSV")
    print("This is an accepted limitation - float64 is compatible with all analysis tools")
    
    return df_converted

def transform_production_to_long_format(df, date_column='date'):
    """
    Transform production data to long format.
    IMPORTANT: Only creates MONTHLY rows (no daily/weekly as per architecture decision).
    
    Args:
        df (DataFrame): Production data with standardized columns
        date_column (str): Name of date column
    
    Returns:
        DataFrame: Production data in long format (monthly only)
    """
    print("\nTRANSFORMING TO LONG FORMAT:")
    print("-" * 40)
    print("ARCHITECTURAL NOTE: Creating MONTHLY rows only")
    print("No daily or weekly rows will be created for production mix data")
    print("Consistent with climate data architecture")
    print()
    
    df_long = df.copy()
    
    # Ensure date is datetime
    df_long[date_column] = pd.to_datetime(df_long[date_column])
    
    # Add time components
    df_long['year'] = df_long[date_column].dt.year
    df_long['month'] = df_long[date_column].dt.month
    df_long['quarter'] = df_long[date_column].dt.quarter
    df_long['week'] = df_long[date_column].dt.isocalendar().week
    df_long['month_name'] = df_long[date_column].dt.month_name()
    
    # Add aggregation level - MONTHLY ONLY
    df_long['aggregation_level'] = 'monthly'
    
    # Convert date to first of month in ISO format (YYYY-MM-DD)
    df_long['date'] = df_long[date_column].apply(lambda x: x.replace(day=1).strftime('%Y-%m-%d'))
    
    # Select final columns in correct order
    base_columns = ['date', 'year', 'month', 'quarter', 'week', 'aggregation_level', 'month_name']
    prod_columns = [col for col in df_long.columns if col.startswith('prod_')]
    final_columns = base_columns + prod_columns
    
    df_final = df_long[final_columns].copy()
    
    print(f"FINAL STRUCTURE:")
    print(f"  Rows: {len(df_final)} (monthly only)")
    print(f"  Columns: {len(df_final.columns)}")
    print(f"  Production variables: {len(prod_columns)}")
    print(f"  Date range: {df_final['date'].min()} to {df_final['date'].max()}")
    print()
    
    print("Production columns:")
    for col in prod_columns:
        print(f"  {col}")
    print()
    
    print("Sample data:")
    print(df_final.head())
    
    return df_final

# Execute data type conversion and transformation
if 'production_df' in locals() and production_df is not None:
    production_converted = convert_production_data_types(production_df)
    
    print(f"\nData types after conversion:")
    for col in production_converted.columns:
        if col.startswith('prod_'):
            print(f"  {col}: {production_converted[col].dtype}")
    
    production_long_format = transform_production_to_long_format(production_converted)
    
    print("\nLONG FORMAT TRANSFORMATION SUCCESSFUL")
    print(f"  Shape: {production_long_format.shape}")
else:
    print("Skipping transformation - production data not available")

print("Cell 5: Convert Data Types and Create Long Format - COMPLETE")

DATA TYPE CONVERSION:
----------------------------------------
Converting 18 production columns to numeric:
  prod_gross_electricity_production: Int64
  prod_gross_electricity_consumption: Int64
  prod_hydropower_production_total: Int64
  prod_fossil_sk_production: Int64
  prod_fossil_DvfB_production: Int64
  prod_fossil_DvOe_production: Int64
  prod_fossil_gas_production: Int64
  prod_fossil_subtotal_production: Int64
  prod_renewable_bio_production: Int64
  prod_renewable_SoBio_production: Int64
  prod_other_fuels_production: Int64
  prod_fuel_production_total: Int64
  prod_wind_total: Int64
  prod_pv_total: Int64
  prod_geothermal_total: Int64
  prod_power_production_total: Int64
  prod_electricity_imports: Int64
  prod_electricity_exports: Int64

NOTE: Int64 types will convert to float64 when saved to CSV
This is an accepted limitation - float64 is compatible with all analysis tools

Data types after conversion:
  prod_gross_electricity_production: Int64
  prod_gross_electricity_co

In [43]:
# =============================================================================
# CELL 6: Save Production Mix Dataset
# =============================================================================

def save_production_dataset(df, output_dir, filename="production_consolidated.csv"):
    """
    Save production mix dataset as separate validation sample.
    
    Args:
        df (DataFrame): Final production mix dataset
        output_dir (Path): Directory for outputs
        filename (str): Output filename
    
    Returns:
        Path: Path to saved file
    """
    if df is None or len(df) == 0:
        print("No production data to save!")
        return None
    
    output_path = output_dir / filename
    df.to_csv(output_path, index=False)
    
    print(f"PRODUCTION MIX DATASET SAVED:")
    print(f"  Path: {output_path}")
    print(f"  Rows: {len(df)}")
    print(f"  Columns: {len(df.columns)}")
    print(f"  Aggregation level: {df['aggregation_level'].unique()}")
    print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
    
    # Show production columns summary
    prod_columns = [col for col in df.columns if col.startswith('prod_')]
    print(f"  Production variables: {len(prod_columns)}")
    
    return output_path

# Save production dataset as validation sample
if 'production_long_format' in locals() and production_long_format is not None:
    production_file_path = save_production_dataset(production_long_format, PROCESSED_DATA_PATH)
    
    if production_file_path:
        print("\nPRODUCTION MIX FORK SUCCESSFUL")
        print("Ready for merge with data_consolidated.csv")
else:
    print("Skipping save - production_long_format not available")

print("Cell 6: Save Production Mix Dataset - COMPLETE")

PRODUCTION MIX DATASET SAVED:
  Path: c:\Users\paulr\Desktop\Capstone\data\processed\production_consolidated.csv
  Rows: 187
  Columns: 25
  Aggregation level: ['monthly']
  Date range: 2010-01-01 to 2025-07-01
  Production variables: 18

PRODUCTION MIX FORK SUCCESSFUL
Ready for merge with data_consolidated.csv
Cell 6: Save Production Mix Dataset - COMPLETE


In [44]:
# =============================================================================
# CELL 7: Merge Production Mix with Consolidated Data
# =============================================================================

def merge_production_with_consolidated_data(production_df, consolidated_file_path, output_dir):
    """
    Merge production mix data with existing consolidated data (electricity + carbon + climate).
    
    Args:
        production_df (DataFrame): Production dataset in long format (monthly only)
        consolidated_file_path (str/Path): Path to data_consolidated.csv
        output_dir (Path): Directory for final output
    
    Returns:
        DataFrame: Merged dataset with production data added
    """
    consolidated_path = Path(consolidated_file_path)
    
    if not consolidated_path.exists():
        print(f"ERROR: Consolidated file not found at {consolidated_path}")
        return pd.DataFrame()
    
    print("MERGING PRODUCTION MIX WITH CONSOLIDATED DATA:")
    print("-" * 50)
    
    try:
        # Load existing consolidated data (electricity + carbon + climate)
        consolidated_df = pd.read_csv(consolidated_path)
        
        print(f"Existing consolidated data loaded:")
        print(f"  Shape: {consolidated_df.shape}")
        print(f"  Aggregation levels: {consolidated_df['aggregation_level'].value_counts().to_dict()}")
        print()
        
        print(f"Production data to merge:")
        print(f"  Shape: {production_df.shape}")
        print(f"  Aggregation levels: {production_df['aggregation_level'].value_counts().to_dict()}")
        print(f"  Date range: {production_df['date'].min()} to {production_df['date'].max()}")
        print()
        
        # Merge on date and aggregation_level
        # IMPORTANT: Only monthly rows in production will find matches
        merged_df = pd.merge(
            consolidated_df, 
            production_df, 
            on=['date', 'aggregation_level'], 
            how='outer',  # Keep all rows from both datasets
            suffixes=('', '_production')
        )
        
        print(f"After merge:")
        print(f"  Shape: {merged_df.shape}")
        print()
        
        # Handle duplicate columns from merge
        duplicate_cols = ['year', 'month', 'quarter', 'week', 'month_name']
        for col in duplicate_cols:
            production_col = f"{col}_production"
            if production_col in merged_df.columns:
                # Use existing values, fill missing with production values
                merged_df[col] = merged_df[col].fillna(merged_df[production_col])
                merged_df.drop(production_col, axis=1, inplace=True)
                print(f"  Resolved duplicate: {col}")
        
        # Sort by date and aggregation level
        merged_df = merged_df.sort_values(['date', 'aggregation_level']).reset_index(drop=True)
        
        # Save merged dataset (overwrites existing data_consolidated.csv)
        final_path = output_dir / "data_consolidated.csv"
        merged_df.to_csv(final_path, index=False)
        
        print(f"\nFINAL CONSOLIDATED DATASET:")
        print(f"  Saved to: {final_path}")
        print(f"  Final shape: {merged_df.shape}")
        print(f"  Date range: {merged_df['date'].min()} to {merged_df['date'].max()}")
        print(f"  Aggregation levels: {merged_df['aggregation_level'].value_counts().to_dict()}")
        print()
        
        # Show which columns are production-specific
        production_columns = [col for col in merged_df.columns if col.startswith('prod_')]
        print(f"Production columns added ({len(production_columns)}):")
        for col in production_columns:
            non_null = merged_df[col].notna().sum()
            pct_missing = (merged_df[col].isna().sum() / len(merged_df)) * 100
            print(f"  {col}: {non_null} non-null values ({pct_missing:.1f}% missing)")
        print()
        
        print("Sample of merged data (showing first 5 production columns):")
        sample_cols = ['date', 'aggregation_level'] + production_columns[:5]
        print(merged_df[sample_cols].head(10))
        
        return merged_df
        
    except Exception as e:
        print(f"ERROR during merge: {e}")
        return pd.DataFrame()

# Execute merge with consolidated data
if 'production_long_format' in locals() and production_long_format is not None:
    consolidated_file = PROCESSED_DATA_PATH / "data_consolidated.csv"
    final_merged_df = merge_production_with_consolidated_data(
        production_long_format, 
        consolidated_file, 
        PROCESSED_DATA_PATH
    )
    
    if len(final_merged_df) > 0:
        print("\nPRODUCTION MIX MERGE SUCCESSFUL")
        print(f"data_consolidated.csv now contains: Electricity + Carbon + Climate + Production")
    else:
        print("\nMERGE FAILED")
else:
    print("Skipping merge - production_long_format not available")

print("Cell 7: Merge Production Mix with Consolidated Data - COMPLETE")

MERGING PRODUCTION MIX WITH CONSOLIDATED DATA:
--------------------------------------------------
Existing consolidated data loaded:
  Shape: (4622, 19)
  Aggregation levels: {'daily': 3896, 'weekly': 566, 'monthly': 160}

Production data to merge:
  Shape: (187, 25)
  Aggregation levels: {'monthly': 187}
  Date range: 2010-01-01 to 2025-07-01

After merge:
  Shape: (4650, 42)

  Resolved duplicate: year
  Resolved duplicate: month
  Resolved duplicate: quarter
  Resolved duplicate: week
  Resolved duplicate: month_name

FINAL CONSOLIDATED DATASET:
  Saved to: c:\Users\paulr\Desktop\Capstone\data\processed\data_consolidated.csv
  Final shape: (4650, 37)
  Date range: 2010-01-01 to 2025-09-01
  Aggregation levels: {'daily': 3896, 'weekly': 566, 'monthly': 188}

Production columns added (18):
  prod_gross_electricity_production: 187 non-null values (96.0% missing)
  prod_gross_electricity_consumption: 187 non-null values (96.0% missing)
  prod_hydropower_production_total: 187 non-null va

In [46]:
# =============================================================================
# CELL 8: Comprehensive Diagnostics for Final Consolidated Dataset
# =============================================================================

def run_comprehensive_diagnostics(consolidated_file_path):
    """
    Run comprehensive quality control diagnostics on final consolidated dataset.
    Verifies data integrity, completeness, and structure across all data sources.
    
    Args:
        consolidated_file_path (Path): Path to data_consolidated.csv
    
    Returns:
        DataFrame: Loaded consolidated dataset for further inspection
    """
    consolidated_path = Path(consolidated_file_path)
    
    if not consolidated_path.exists():
        print(f"ERROR: Consolidated file not found at {consolidated_path}")
        return None
    
    print("="*70)
    print("COMPREHENSIVE DIAGNOSTICS: data_consolidated.csv")
    print("="*70)
    print()
    
    try:
        df = pd.read_csv(consolidated_path)
        
        # =====================================================================
        # 1. OVERALL STRUCTURE
        # =====================================================================
        print("1. OVERALL STRUCTURE:")
        print("-" * 70)
        print(f"  Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"  Date range (overall): {df['date'].min()} to {df['date'].max()}")
        print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
        print()
        
        # =====================================================================
        # 2. AGGREGATION LEVEL DISTRIBUTION
        # =====================================================================
        print("2. AGGREGATION LEVEL DISTRIBUTION:")
        print("-" * 70)
        agg_counts = df['aggregation_level'].value_counts().sort_index()
        for level, count in agg_counts.items():
            pct = (count / len(df)) * 100
            date_range_level = df[df['aggregation_level'] == level]['date'].agg(['min', 'max'])
            print(f"  {level:10s}: {count:5d} rows ({pct:5.1f}%) | Range: {date_range_level['min']} to {date_range_level['max']}")
        print()
        
        # =====================================================================
        # 3. COLUMN INVENTORY BY DATA SOURCE
        # =====================================================================
        print("3. COLUMN INVENTORY BY DATA SOURCE:")
        print("-" * 70)
        
        base_cols = ['date', 'year', 'month', 'quarter', 'week', 'aggregation_level', 'month_name']
        electricity_cols = [col for col in df.columns if col.startswith('price_') or col == 'data_completeness']
        carbon_cols = [col for col in df.columns if col.startswith('carbonprices_')]
        climate_cols = [col for col in df.columns if col.startswith('climate_')]
        production_cols = [col for col in df.columns if col.startswith('prod_')]
        
        print(f"  Base structure: {len(base_cols)} columns")
        print(f"  Electricity:    {len(electricity_cols)} columns - {electricity_cols}")
        print(f"  Carbon:         {len(carbon_cols)} columns - {carbon_cols}")
        print(f"  Climate:        {len(climate_cols)} columns - {climate_cols}")
        print(f"  Production:     {len(production_cols)} columns")
        print(f"    (Showing first 5: {production_cols[:5]})")
        print(f"  Total: {len(df.columns)} columns")
        print()
        
        # =====================================================================
        # 4. MISSING VALUES ANALYSIS
        # =====================================================================
        print("4. MISSING VALUES ANALYSIS:")
        print("-" * 70)
        
        def analyze_missing_by_source(columns, source_name):
            if len(columns) == 0:
                print(f"  {source_name}: No columns found")
                return
            
            print(f"  {source_name}:")
            for col in columns:
                missing_count = df[col].isna().sum()
                missing_pct = (missing_count / len(df)) * 100
                print(f"    {col:45s}: {missing_count:5d} missing ({missing_pct:5.1f}%)")
        
        analyze_missing_by_source(electricity_cols, "ELECTRICITY")
        print()
        analyze_missing_by_source(carbon_cols, "CARBON")
        print()
        analyze_missing_by_source(climate_cols, "CLIMATE")
        print()
        
        # Production columns - show summary
        if len(production_cols) > 0:
            print(f"  PRODUCTION (summary of {len(production_cols)} columns):")
            for col in production_cols:
                missing_count = df[col].isna().sum()
                missing_pct = (missing_count / len(df)) * 100
                print(f"    {col:45s}: {missing_count:5d} missing ({missing_pct:5.1f}%)")
        print()
        
        # =====================================================================
        # 5. DATA TYPES VERIFICATION
        # =====================================================================
        print("5. DATA TYPES VERIFICATION:")
        print("-" * 70)
        print("  Expected: All numeric columns should be float64 (due to CSV format)")
        print()
        
        numeric_cols = electricity_cols + carbon_cols + climate_cols + production_cols
        non_float_cols = [col for col in numeric_cols if df[col].dtype != 'float64']
        
        if len(non_float_cols) > 0:
            print(f"  WARNING: {len(non_float_cols)} columns are not float64:")
            for col in non_float_cols:
                print(f"    {col}: {df[col].dtype}")
        else:
            print(f"  ✓ All {len(numeric_cols)} numeric columns are float64")
        print()
        
        # =====================================================================
        # 6. ARCHITECTURAL VALIDATION
        # =====================================================================
        print("6. ARCHITECTURAL VALIDATION:")
        print("-" * 70)
        
        # Check daily rows - should only have electricity + carbon
        daily_df = df[df['aggregation_level'] == 'daily']
        if len(daily_df) > 0:
            daily_climate_nulls = daily_df[climate_cols].isna().all(axis=1).sum()
            daily_prod_nulls = daily_df[production_cols].isna().all(axis=1).sum() if len(production_cols) > 0 else len(daily_df)
            print(f"  Daily rows ({len(daily_df)}):")
            print(f"    Climate columns all null: {daily_climate_nulls}/{len(daily_df)} rows ({daily_climate_nulls/len(daily_df)*100:.1f}%) ✓")
            print(f"    Production columns all null: {daily_prod_nulls}/{len(daily_df)} rows ({daily_prod_nulls/len(daily_df)*100:.1f}%) ✓")
        
        # Check weekly rows - should only have electricity + carbon
        weekly_df = df[df['aggregation_level'] == 'weekly']
        if len(weekly_df) > 0:
            weekly_climate_nulls = weekly_df[climate_cols].isna().all(axis=1).sum()
            weekly_prod_nulls = weekly_df[production_cols].isna().all(axis=1).sum() if len(production_cols) > 0 else len(weekly_df)
            print(f"  Weekly rows ({len(weekly_df)}):")
            print(f"    Climate columns all null: {weekly_climate_nulls}/{len(weekly_df)} rows ({weekly_climate_nulls/len(weekly_df)*100:.1f}%) ✓")
            print(f"    Production columns all null: {weekly_prod_nulls}/{len(weekly_df)} rows ({weekly_prod_nulls/len(weekly_df)*100:.1f}%) ✓")
        
        # Check monthly rows - should have all 4 sources
        monthly_df = df[df['aggregation_level'] == 'monthly']
        if len(monthly_df) > 0:
            monthly_climate_data = monthly_df[climate_cols].notna().any(axis=1).sum()
            monthly_prod_data = monthly_df[production_cols].notna().any(axis=1).sum() if len(production_cols) > 0 else 0
            print(f"  Monthly rows ({len(monthly_df)}):")
            print(f"    Rows with climate data: {monthly_climate_data}/{len(monthly_df)} ({monthly_climate_data/len(monthly_df)*100:.1f}%)")
            print(f"    Rows with production data: {monthly_prod_data}/{len(monthly_df)} ({monthly_prod_data/len(monthly_df)*100:.1f}%)")
        print()
        
        # =====================================================================
        # 7. SAMPLE DATA FROM EACH AGGREGATION LEVEL
        # =====================================================================
        print("7. SAMPLE DATA FROM EACH AGGREGATION LEVEL:")
        print("-" * 70)
        
        for level in ['daily', 'weekly', 'monthly']:
            level_df = df[df['aggregation_level'] == level]
            if len(level_df) > 0:
                print(f"\n  {level.upper()} sample (first 3 rows):")
                # Show subset of columns for readability
                sample_cols = ['date', 'aggregation_level', 'price_exaa_mean', 'carbonprices_primary_market']
                if level == 'monthly':
                    sample_cols.extend(['climate_hdd_at', 'prod_gross_electricity_production'])
                display_cols = [col for col in sample_cols if col in level_df.columns]
                print(level_df[display_cols].head(3).to_string(index=False))
        
        print()
        print("="*70)
        print("DIAGNOSTICS COMPLETE")
        print("="*70)
        
        return df
        
    except Exception as e:
        print(f"ERROR during diagnostics: {e}")
        return None

# Execute comprehensive diagnostics
consolidated_file = PROCESSED_DATA_PATH / "data_consolidated.csv"
final_diagnostics_df = run_comprehensive_diagnostics(consolidated_file)

if final_diagnostics_df is not None:
    print("\n✓ data_consolidated.csv is ready for EDA and modeling")
    print(f"  Contains: Electricity + Carbon + Climate + Production Mix")
    print(f"  Total: {final_diagnostics_df.shape[0]} rows × {final_diagnostics_df.shape[1]} columns")
else:
    print("\n✗ Diagnostics failed - check errors above")

print("Cell 8: Comprehensive Diagnostics - COMPLETE")

COMPREHENSIVE DIAGNOSTICS: data_consolidated.csv

1. OVERALL STRUCTURE:
----------------------------------------------------------------------
  Shape: 4650 rows × 37 columns
  Date range (overall): 2010-01-01 to 2025-09-01
  Memory usage: 1.95 MB

2. AGGREGATION LEVEL DISTRIBUTION:
----------------------------------------------------------------------
  daily     :  3896 rows ( 83.8%) | Range: 2015-01-01 to 2025-08-31
  monthly   :   188 rows (  4.0%) | Range: 2010-01-01 to 2025-08-01
  weekly    :   566 rows ( 12.2%) | Range: 2015-01-05 to 2025-09-01

3. COLUMN INVENTORY BY DATA SOURCE:
----------------------------------------------------------------------
  Base structure: 7 columns
  Electricity:    5 columns - ['price_exaa_mean', 'price_mc_auction_mean', 'price_count_exaa', 'price_count_mc', 'data_completeness']
  Carbon:         3 columns - ['carbonprices_exchange_rate_eur_usd', 'carbonprices_primary_market', 'carbonprices_secondary_market']
  Climate:        4 columns - ['climat